# Notebook 14: Large Blood Cohort Replication & Cross-Cohort Projection

**Dataset:** [GSE18123](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE18123) — Kong et al. 2012, *PLoS ONE*
**Title:** "Characteristics and Predictive Value of Blood Transcriptome Signature in Males with Autism Spectrum Disorders"
**Platform:** Affymetrix Human Genome U133 Plus 2.0 (GPL570) + Human Gene 1.0 ST (GPL6244)
**Samples:** 285 blood samples from males (170 ASD + 115 controls), two cohorts
**Design:** Discovery cohort (P1: 99 samples) + Validation cohort (P2: 186 samples)

**Framework:** [pathway-subtyping](https://pypi.org/project/pathway-subtyping/) v0.3.0
**Author:** Rohit Chauhan ([ORCID: 0009-0003-9895-4629](https://orcid.org/0009-0003-9895-4629))

---

## What this notebook does

1. Downloads GSE18123 blood expression data from GEO (two Affymetrix platforms)
2. Merges expression data across platforms using common gene symbols
3. Runs pathway-level scoring using 15 curated autism pathways
4. Discovers molecular subtypes via GMM clustering
5. Validates subtypes through 3 validation gates
6. Characterizes subtypes (enriched pathways, top genes)
7. **Cross-cohort projection from GSE111175** (KEY — replication test)
8. Cross-tissue comparison with GSE28521 postmortem brain
9. Benchmarks against alternative clustering methods

**Vulnerabilities addressed:**
- **V5** — Sample size (n=285, 9x larger than GSE28521)
- **V7** — Postmortem brain only (second blood dataset confirms blood-based subtyping)

**Runtime:** ~15-20 minutes on Colab Pro

## 1. Setup & Installation

In [ ]:
# Install pathway-subtyping framework with visualization extras
!pip install -q pathway-subtyping[viz]==0.3.1 GEOparse

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Framework imports
from pathway_subtyping import (
    score_pathways_from_expression,
    ExpressionScoringMethod,
    run_clustering,
    ClusteringAlgorithm,
    select_n_clusters,
    compare_algorithms,
    ValidationGates,
    characterize_subtypes,
    generate_subtype_heatmap,
    generate_gene_heatmap,
    export_characterization,
    run_benchmark_comparison,
    compute_dim_reduction,
    DimReductionMethod,
)

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Output directory
OUTPUT_DIR = './outputs/gse18123'
os.makedirs(OUTPUT_DIR, exist_ok=True)

DATA_DIR = './data'
os.makedirs(DATA_DIR, exist_ok=True)

print('Setup complete.')

## 2. Download GSE18123 from GEO

GSE18123 uses two Affymetrix platforms across two cohorts:
- **Discovery (P1):** GPL570 — Affymetrix Human Genome U133 Plus 2.0 Array (99 samples)
- **Validation (P2):** GPL6244 — Affymetrix Human Gene 1.0 ST Array (186 samples)

We download via GEOparse using HTTP (more reliable than FTP).

In [ ]:
import GEOparse
import os as _dl_os
import time

# Force HTTP instead of FTP — NCBI FTP is unreliable
_dl_os.environ['GEOPARSE_USE_HTTP_FOR_FTP'] = 'yes'

soft_file = _dl_os.path.join(DATA_DIR, 'GSE18123_family.soft.gz')
if _dl_os.path.exists(soft_file):
    print(f'Using cached SOFT file: {soft_file}')
    gse = GEOparse.get_GEO(filepath=soft_file, silent=True)
else:
    for attempt in range(1, 4):
        try:
            print(f'Downloading GSE18123 from GEO (attempt {attempt}/3, using HTTP)...')
            gse = GEOparse.get_GEO(geo='GSE18123', destdir=DATA_DIR, silent=True)
            print('Download successful.')
            break
        except Exception as e:
            print(f'Attempt {attempt} failed: {e}')
            for f in _dl_os.listdir(DATA_DIR):
                if f.startswith('GSE18123') and f.endswith('.tmp'):
                    _dl_os.remove(_dl_os.path.join(DATA_DIR, f))
            if attempt < 3:
                wait = 10 * attempt
                print(f'Retrying in {wait}s...')
                time.sleep(wait)
            else:
                raise RuntimeError(
                    f'Failed to download GSE18123 after 3 attempts. '
                    f'Try downloading manually from '
                    f'https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE18123 '
                    f'and placing the .soft.gz file in {DATA_DIR}/'
                ) from e

print(f'Platform(s): {list(gse.gpls.keys())}')
print(f'Number of samples: {len(gse.gsms)}')

# Show platform breakdown
platform_counts = {}
for gsm_name, gsm in gse.gsms.items():
    gpl = gsm.metadata.get('platform_id', ['unknown'])[0]
    platform_counts[gpl] = platform_counts.get(gpl, 0) + 1
print(f'\nSamples per platform:')
for gpl, count in sorted(platform_counts.items()):
    print(f'  {gpl}: {count} samples')

## 3. Extract Sample Metadata

Parse sample characteristics to extract diagnosis, age, sex, and cohort assignment.

In [ ]:
# Extract all metadata from sample characteristics
metadata_rows = []
for gsm_name, gsm in gse.gsms.items():
    chars = gsm.metadata.get('characteristics_ch1', [])
    char_dict = {}
    for c in chars:
        if ':' in c:
            key, val = c.split(':', 1)
            char_dict[key.strip().lower()] = val.strip()

    title = gsm.metadata.get('title', [''])[0]
    source = gsm.metadata.get('source_name_ch1', [''])[0]
    platform = gsm.metadata.get('platform_id', [''])[0]

    metadata_rows.append({
        'sample_id': gsm_name,
        'title': title,
        'source': source,
        'platform': platform,
        **char_dict,
    })

metadata = pd.DataFrame(metadata_rows).set_index('sample_id')
print(f'Metadata columns: {list(metadata.columns)}')
print(f'Total samples: {len(metadata)}')
metadata.head()

In [ ]:
# Parse diagnosis from metadata
diag_col = None
for col in metadata.columns:
    if any(kw in col.lower() for kw in ['diagnosis', 'disease', 'status', 'group', 'condition']):
        diag_col = col
        break

if diag_col is None:
    # Try to infer from title or source
    print('No explicit diagnosis column. Checking title/source...')
    print(f'Unique sources: {metadata["source"].unique()[:10]}')
    print(f'Sample titles (first 5): {metadata["title"].unique()[:5]}')
    # Infer from title pattern: titles starting with 'A' = ASD, 'M'/'F' = control
    if metadata['title'].str.match(r'^[AMF]').all():
        metadata['diagnosis'] = metadata['title'].apply(
            lambda x: 'ASD' if x.startswith('A') else 'Control'
        )
        diag_col = 'diagnosis'
        print('Inferred diagnosis from title prefix (A=ASD, M/F=Control)')
    elif metadata['source'].str.contains('autism|asd', case=False).any():
        metadata['diagnosis'] = metadata['source'].apply(
            lambda x: 'ASD' if 'asd' in x.lower() or 'autism' in x.lower() else 'Control'
        )
        diag_col = 'diagnosis'
else:
    metadata['diagnosis'] = metadata[diag_col].apply(
        lambda x: 'ASD' if 'asd' in str(x).lower() or 'autism' in str(x).lower() else 'Control'
    )

print(f'\nDiagnosis column: {diag_col}')
print(f'\n--- Sample Breakdown ---')
print(metadata['diagnosis'].value_counts())

# Cross-tabulate with platform
print(f'\n--- Diagnosis × Platform ---')
print(pd.crosstab(metadata['diagnosis'], metadata['platform'], margins=True))

## 4. Build Expression Matrix

GSE18123 uses two Affymetrix platforms. We handle this by:
1. Extracting probe-level data from each platform separately
2. Mapping probes to gene symbols using platform annotations
3. Collapsing to gene-level (mean of probes per gene)
4. Merging on common gene symbols across platforms

This yields a unified (samples × genes) matrix across both cohorts.

In [ ]:
# Extract expression data per platform and map to gene symbols
platform_gene_expr = {}  # {gpl_id: DataFrame (samples x genes)}

for gpl_id, gpl in gse.gpls.items():
    print(f'\n=== Processing {gpl_id} ===')
    gpl_table = gpl.table

    # Find gene symbol column
    symbol_col = None
    for col in ['Gene Symbol', 'Symbol', 'GENE_SYMBOL', 'Gene_Symbol', 'gene_assignment']:
        if col in gpl_table.columns:
            symbol_col = col
            break
    if symbol_col is None:
        for col in gpl_table.columns:
            if 'symbol' in col.lower() or 'gene' in col.lower():
                symbol_col = col
                break

    print(f'  Platform: {gpl.metadata.get("title", ["Unknown"])[0]}')
    print(f'  Gene symbol column: "{symbol_col}"')

    # Get samples for this platform
    platform_samples = [s for s, gsm in gse.gsms.items()
                        if gsm.metadata.get('platform_id', [''])[0] == gpl_id]
    print(f'  Samples: {len(platform_samples)}')

    if len(platform_samples) == 0 or symbol_col is None:
        print(f'  SKIPPING — no samples or no gene symbol column')
        continue

    # Build probe-level expression matrix for this platform
    probe_dfs = []
    for gsm_name in platform_samples:
        gsm = gse.gsms[gsm_name]
        tbl = gsm.table
        if tbl is not None and 'VALUE' in tbl.columns and 'ID_REF' in tbl.columns:
            s = tbl.set_index('ID_REF')['VALUE']
            s.name = gsm_name
            probe_dfs.append(s)

    if not probe_dfs:
        print(f'  SKIPPING — no expression data found')
        continue

    expr_platform = pd.concat(probe_dfs, axis=1)
    expr_platform = expr_platform.apply(pd.to_numeric, errors='coerce')
    expr_platform = expr_platform.dropna(how='all')
    print(f'  Raw: {expr_platform.shape[0]} probes x {expr_platform.shape[1]} samples')

    # Map probes to gene symbols
    probe_to_gene = gpl_table.set_index('ID')[symbol_col].dropna()

    # Handle 'gene_assignment' column (GPL6244) which has format: 'chr // gene // ...' 
    if symbol_col == 'gene_assignment':
        probe_to_gene = probe_to_gene.apply(
            lambda x: str(x).split('//')[1].strip() if '//' in str(x) else ''
        )
    # Handle 'Gene Symbol' (GPL570) which may have '///' separators
    else:
        probe_to_gene = probe_to_gene.apply(
            lambda x: str(x).split('///')[0].strip()
        )

    probe_to_gene = probe_to_gene[probe_to_gene.str.strip() != '']
    probe_to_gene = probe_to_gene[probe_to_gene != '---']
    print(f'  Probes with gene symbols: {len(probe_to_gene)}')

    # Map and collapse
    common_probes = expr_platform.index.intersection(probe_to_gene.index)
    expr_mapped = expr_platform.loc[common_probes].copy()
    expr_mapped['gene_symbol'] = probe_to_gene.loc[common_probes].values
    gene_expr = expr_mapped.groupby('gene_symbol').mean()
    gene_expr = gene_expr.T  # samples x genes
    print(f'  Gene-level: {gene_expr.shape[0]} samples x {gene_expr.shape[1]} genes')

    platform_gene_expr[gpl_id] = gene_expr

print(f'\nPlatforms processed: {list(platform_gene_expr.keys())}')

In [ ]:
# Merge expression data across platforms using common genes
if len(platform_gene_expr) == 1:
    gpl_id = list(platform_gene_expr.keys())[0]
    gene_expression = platform_gene_expr[gpl_id]
    print(f'Single platform ({gpl_id}): {gene_expression.shape}')
elif len(platform_gene_expr) >= 2:
    gpl_ids = sorted(platform_gene_expr.keys())
    genes_per_platform = [set(platform_gene_expr[g].columns) for g in gpl_ids]
    common_genes = genes_per_platform[0]
    for gs in genes_per_platform[1:]:
        common_genes = common_genes & gs
    common_genes = sorted(common_genes)
    print(f'Genes per platform: {[len(g) for g in genes_per_platform]}')
    print(f'Common genes across all platforms: {len(common_genes)}')

    # Concatenate on common genes
    dfs = [platform_gene_expr[g][common_genes] for g in gpl_ids]
    gene_expression = pd.concat(dfs, axis=0)
    print(f'\nMerged expression matrix: {gene_expression.shape[0]} samples x {gene_expression.shape[1]} genes')

    # Add platform info to metadata
    print(f'\nSamples per platform in merged matrix:')
    for gpl_id in gpl_ids:
        n = len(platform_gene_expr[gpl_id])
        print(f'  {gpl_id}: {n} samples')
else:
    raise ValueError('No platform data extracted')

# Log2 transform check
max_val = gene_expression.max().max()
print(f'\nMax expression value: {max_val:.2f}')
if max_val > 30:
    print('Data appears raw — applying log2(x+1) transform')
    gene_expression = np.log2(gene_expression.clip(lower=0) + 1)
else:
    print('Data appears already log-transformed')

In [ ]:
# Expression matrix QC
print('--- Expression Matrix QC ---')
print(f'Shape: {gene_expression.shape}')
print(f'Missing values: {gene_expression.isna().sum().sum()}')
print(f'Expression range: [{gene_expression.min().min():.2f}, {gene_expression.max().max():.2f}]')
print(f'Mean expression: {gene_expression.mean().mean():.2f}')

# Drop genes with zero variance
gene_var = gene_expression.var()
n_zero_var = (gene_var == 0).sum()
if n_zero_var > 0:
    gene_expression = gene_expression.loc[:, gene_var > 0]
    print(f'Dropped {n_zero_var} zero-variance genes. Remaining: {gene_expression.shape[1]}')

# Fill NaN with column median
if gene_expression.isna().any().any():
    n_nan = gene_expression.isna().sum().sum()
    gene_expression = gene_expression.fillna(gene_expression.median())
    print(f'Filled {n_nan} NaN values with column medians.')

# Align metadata with expression
common_samples = gene_expression.index.intersection(metadata.index)
gene_expression = gene_expression.loc[common_samples]
metadata = metadata.loc[common_samples]
print(f'\nFinal: {gene_expression.shape[0]} samples x {gene_expression.shape[1]} genes')
print(f'Metadata aligned: {len(metadata)} samples')

# Diagnosis breakdown after alignment
print(f'\n--- Final Sample Breakdown ---')
print(metadata['diagnosis'].value_counts())

In [ ]:
# Visualize potential platform batch effect using PCA
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=SEED)
pca_coords = pca.fit_transform(gene_expression.values)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Color by platform
platforms = metadata.loc[gene_expression.index, 'platform']
for plat, color in zip(sorted(platforms.unique()), ['#e74c3c', '#3498db', '#2ecc71']):
    mask = platforms == plat
    axes[0].scatter(pca_coords[mask, 0], pca_coords[mask, 1], c=color,
                    label=f'{plat} (n={mask.sum()})', s=40, alpha=0.6,
                    edgecolors='k', linewidth=0.3)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].set_title('Colored by Platform')
axes[0].legend(fontsize=9)

# Right: Color by diagnosis
for dx, color in [('ASD', 'coral'), ('Control', 'steelblue')]:
    mask = metadata.loc[gene_expression.index, 'diagnosis'] == dx
    axes[1].scatter(pca_coords[mask.values, 0], pca_coords[mask.values, 1], c=color,
                    label=f'{dx} (n={mask.sum()})', s=40, alpha=0.6,
                    edgecolors='k', linewidth=0.3)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].set_title('Colored by Diagnosis')
axes[1].legend(fontsize=9)

plt.suptitle('GSE18123: PCA of Gene Expression (Platform Batch Check)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'platform_batch_pca.png'), dpi=150, bbox_inches='tight')
plt.show()

# Check if platform effect is dominant
print(f'PC1 explains {pca.explained_variance_ratio_[0]*100:.1f}% of variance')
print(f'PC2 explains {pca.explained_variance_ratio_[1]*100:.1f}% of variance')
print('\nNote: If platform clusters dominate PC1/PC2, pathway-level scoring')
print('may mitigate this since pathways aggregate across many genes.')

## 5. Load Autism Pathway Gene Sets & Score

The framework ships with 15 curated autism pathway gene sets derived from
SFARI Gene, Satterstrom et al. 2020, and ASC exome studies.

We use ssGSEA (single-sample Gene Set Enrichment Analysis) to reduce
the gene expression matrix to a 15-pathway score matrix. Pathway-level
scoring also helps mitigate platform batch effects since pathways aggregate
signal across many genes.

In [ ]:
import urllib.request

GMT_URL = 'https://raw.githubusercontent.com/topmist-admin/pathway-subtyping-framework/main/data/pathways/autism_pathways.gmt'
GMT_PATH = os.path.join(DATA_DIR, 'autism_pathways.gmt')

if not os.path.exists(GMT_PATH):
    urllib.request.urlretrieve(GMT_URL, GMT_PATH)
    print(f'Downloaded autism_pathways.gmt')

# Parse GMT file
pathways = {}
with open(GMT_PATH) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        parts = line.split('\t')
        if len(parts) >= 3:
            pathways[parts[0]] = parts[2:]

print(f'Loaded {len(pathways)} pathways:')
total_genes = set()
for name, genes in pathways.items():
    available = len(set(genes) & set(gene_expression.columns))
    total_genes.update(genes)
    print(f'  {name}: {len(genes)} genes ({available} found in expression data)')

print(f'\nTotal unique pathway genes: {len(total_genes)}')
print(f'Found in expression data: {len(total_genes & set(gene_expression.columns))}')

In [ ]:
# Score pathways using ssGSEA
scoring_result = score_pathways_from_expression(
    gene_expression=gene_expression,
    pathways=pathways,
    method=ExpressionScoringMethod.SSGSEA,
    min_genes_per_pathway=2,
    seed=SEED,
    show_progress=True,
)

pathway_scores = scoring_result.pathway_scores

print('\n--- Scoring Report ---')
print(scoring_result.format_report())
print(f'\nPathway score matrix: {pathway_scores.shape}')
print(f'Pathways scored: {scoring_result.n_pathways_scored}')
if scoring_result.skipped_pathways:
    print(f'Skipped: {scoring_result.skipped_pathways}')

In [ ]:
# Visualize pathway score distribution by diagnosis
scores_with_meta = pathway_scores.copy()
scores_with_meta['diagnosis'] = metadata.loc[pathway_scores.index, 'diagnosis']

n_pathways = pathway_scores.shape[1]
n_cols = 5
n_rows = (n_pathways + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
axes_flat = axes.flatten()

for i, pathway in enumerate(pathway_scores.columns):
    ax = axes_flat[i]
    for dx, color in [('ASD', 'coral'), ('Control', 'steelblue')]:
        subset = scores_with_meta[scores_with_meta['diagnosis'] == dx][pathway]
        ax.hist(subset, alpha=0.6, label=dx, color=color, bins=15)
    ax.set_title(pathway.replace('_', '\n'), fontsize=8)
    if i == 0:
        ax.legend(fontsize=7)

for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('GSE18123 Blood: Pathway Score Distributions (ASD vs Control)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'pathway_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Optimal Cluster Selection

Use BIC (Bayesian Information Criterion) to select the optimal number of molecular subtypes.
We test k=2 through k=7 on the ASD samples only — controls serve as a reference population.

In [ ]:
# Subset to ASD samples for subtype discovery
asd_mask = metadata['diagnosis'] == 'ASD'
asd_scores = pathway_scores.loc[asd_mask]
asd_expression = gene_expression.loc[asd_mask]
asd_meta = metadata.loc[asd_mask].copy()

print(f'ASD samples for subtype discovery: {len(asd_scores)}')
print(f'Control samples (reference): {(~asd_mask).sum()}')

# Select optimal number of clusters using BIC
selection = select_n_clusters(
    data=asd_scores.values,
    k_range=list(range(2, 8)),
    method='bic',
    seed=SEED,
)

optimal_k = selection.optimal_k
print(f'\nOptimal k (BIC): {optimal_k}')
print(f'\nBIC values: {selection.bic_values}')
print(f'Silhouette values: {selection.silhouette_values}')

In [ ]:
# Plot BIC and Silhouette curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ks = sorted(selection.bic_values.keys())
ax1.plot(ks, [selection.bic_values[k] for k in ks], 'bo-', linewidth=2)
ax1.axvline(x=optimal_k, color='red', linestyle='--', label=f'Optimal k={optimal_k}')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('BIC (lower is better)')
ax1.set_title('Model Selection: BIC')
ax1.legend()

ax2.plot(ks, [selection.silhouette_values[k] for k in ks], 'go-', linewidth=2)
ax2.axvline(x=optimal_k, color='red', linestyle='--', label=f'Optimal k={optimal_k}')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score (higher is better)')
ax2.set_title('Model Selection: Silhouette')
ax2.legend()

plt.suptitle('GSE18123 Blood (ASD only): Optimal Cluster Selection',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model_selection.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. GMM Clustering & Validation

Run Gaussian Mixture Model clustering at the BIC-optimal k on ASD samples,
then validate through 3 gates:
1. **Negative Control 1 (Label Shuffle):** Shuffled labels should NOT be recoverable
2. **Negative Control 2 (Random Gene Sets):** Random pathways should NOT reproduce the clusters
3. **Stability (Bootstrap):** Clusters should survive resampling

In [ ]:
# Run GMM clustering on ASD samples
clustering = run_clustering(
    data=asd_scores.values,
    n_clusters=optimal_k,
    algorithm=ClusteringAlgorithm.GMM,
    seed=SEED,
)

print(f'--- GMM Clustering Results (ASD only) ---')
print(f'k = {clustering.n_clusters}')
print(f'Silhouette score: {clustering.silhouette:.4f}')
print(f'Calinski-Harabasz: {clustering.calinski_harabasz:.2f}')
print(f'Davies-Bouldin: {clustering.davies_bouldin:.4f}')
if clustering.bic is not None:
    print(f'BIC: {clustering.bic:.2f}')
print(f'Converged: {clustering.converged}')

labels = clustering.labels
asd_meta['subtype'] = labels

print(f'\nSubtype sizes:')
for i in range(optimal_k):
    count = (labels == i).sum()
    print(f'  Subtype {i}: {count} samples ({count/len(labels)*100:.1f}%)')

# Cross-tabulate with platform
print(f'\n--- Subtype × Platform ---')
print(pd.crosstab(asd_meta['subtype'], asd_meta['platform'], margins=True))

In [ ]:
# PCA scatter plot colored by subtype
embedding, pca_meta = compute_dim_reduction(
    pathway_scores=asd_scores,
    method=DimReductionMethod.PCA,
    n_components=2,
    seed=SEED,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Color by subtype
scatter_colors = plt.cm.Set2(np.linspace(0, 1, optimal_k))
for i in range(optimal_k):
    mask = labels == i
    axes[0].scatter(embedding[mask, 0], embedding[mask, 1], c=[scatter_colors[i]],
                    label=f'Subtype {i} (n={mask.sum()})', s=60, alpha=0.7,
                    edgecolors='k', linewidth=0.5)
axes[0].set_xlabel(f'PC1 ({pca_meta["explained_variance_ratio"][0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca_meta["explained_variance_ratio"][1]*100:.1f}%)')
axes[0].set_title('ASD Blood Subtypes')
axes[0].legend()

# Right: Color by platform
plat_colors = {'GPL570': '#e74c3c', 'GPL6244': '#3498db'}
for plat in sorted(asd_meta['platform'].unique()):
    mask = asd_meta['platform'].values == plat
    c = plat_colors.get(plat, '#2ecc71')
    axes[1].scatter(embedding[mask, 0], embedding[mask, 1], c=c,
                    label=f'{plat} (n={mask.sum()})', s=60, alpha=0.7,
                    edgecolors='k', linewidth=0.5)
axes[1].set_xlabel(f'PC1 ({pca_meta["explained_variance_ratio"][0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca_meta["explained_variance_ratio"][1]*100:.1f}%)')
axes[1].set_title('Colored by Platform')
axes[1].legend()

plt.suptitle(f'GSE18123 Blood: Pathway-Based Molecular Subtypes (k={optimal_k}, GMM)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'pca_scatter_subtypes.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Run all validation gates
gates = ValidationGates(
    seed=SEED,
    n_permutations=200,
    n_bootstrap=100,
    stability_threshold=0.8,
    null_ari_max=0.15,
    show_progress=True,
)

print('Running validation gates (this may take 2-3 minutes)...')
val_result = gates.run_all(
    pathway_scores=asd_scores,
    cluster_labels=labels,
    pathways=pathways,
    gene_burdens=asd_expression,
    n_clusters=optimal_k,
    gmm_seed=SEED,
)

print('\n' + '=' * 60)
print('VALIDATION GATES RESULTS')
print('=' * 60)
print(f'\nAll gates passed: {"YES" if val_result.all_passed else "NO"}')
print()
for gate in val_result.results:
    status = 'PASS' if gate.passed else 'FAIL'
    print(f'  [{status}] {gate.name}: {gate.metric_name} = {gate.metric_value:.4f} '
          f'(threshold: {gate.comparison} {gate.threshold:.4f})')

## 8. Subtype Characterization

Identify which pathways and genes drive each molecular subtype.
This reveals the biological signature of each blood-based ASD subtype.

In [ ]:
# Characterize subtypes
char_result = characterize_subtypes(
    pathway_scores=asd_scores,
    cluster_labels=labels,
    gene_burdens=asd_expression,
    pathways=pathways,
    fdr_alpha=0.05,
    top_n_genes=20,
    seed=SEED,
)

print(char_result.format_report())

In [ ]:
# Generate heatmaps
fig_heatmap = generate_subtype_heatmap(
    char_result,
    output_path=os.path.join(OUTPUT_DIR, 'subtype_heatmap.png'),
    figsize=(14, 8),
)
plt.show()

fig_genes = generate_gene_heatmap(
    char_result,
    output_path=os.path.join(OUTPUT_DIR, 'gene_heatmap.png'),
    figsize=(16, 10),
    top_n=15,
)
plt.show()

In [ ]:
# Export characterization data to CSV
export_files = export_characterization(
    char_result,
    output_dir=OUTPUT_DIR,
    formats=['csv'],
)
print('Exported characterization files:')
for f in export_files:
    print(f'  {f}')

## 9. Cross-Cohort Projection from GSE111175 — KEY REPLICATION TEST

**This is the central analysis addressing Vulnerability V5** (underpowered / replicability).

We test whether molecular subtypes discovered independently in GSE111175 (Notebook 13)
can be projected onto GSE18123 samples and recover similar structure. A high ARI between
independent discovery and projected subtypes demonstrates that blood-based subtypes are
**reproducible across independent cohorts**.

**Method:**
1. Load GSE111175 GMM centroids (fitted model parameters)
2. Score GSE18123 ASD samples on the same pathways
3. Predict subtype assignments using GSE111175 centroids
4. Compare projected labels vs de novo labels using ARI

**Success criterion:** Cross-cohort projection ARI > 0.3

In [ ]:
# Load GSE111175 pathway scores and subtype assignments for centroid computation
import os as _os

has_gse111175 = False

# Search paths (local execution, Colab, GitHub)
search_paths = [
    ('research-results/GSE111175', 'local research-results'),
    ('../../research-results/GSE111175', 'parent research-results'),
    ('./outputs/gse111175', 'local outputs'),
]

for base_path, source_label in search_paths:
    scores_path = _os.path.join(base_path, 'pathway_scores_asd.csv')
    meta_path = _os.path.join(base_path, 'sample_metadata_with_subtypes.csv')
    try:
        gse111175_scores = pd.read_csv(scores_path, index_col=0)
        gse111175_meta = pd.read_csv(meta_path, index_col=0)
        print(f'Loaded GSE111175 results from {source_label}:')
        print(f'  ASD samples: {len(gse111175_scores)}')
        print(f'  Pathways: {gse111175_scores.shape[1]}')
        print(f'  Subtypes: {gse111175_meta["subtype"].nunique()}')
        print(f'  Subtype sizes: {gse111175_meta["subtype"].value_counts().to_dict()}')
        has_gse111175 = True
        break
    except Exception:
        continue

if not has_gse111175:
    print('WARNING: Could not load GSE111175 results.')
    print('Run Notebook 13 first to generate outputs, then copy to research-results/GSE111175/')
    print('Skipping cross-cohort projection.')

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score

if has_gse111175:
    # Find shared pathways between GSE111175 and GSE18123
    shared_pathways = sorted(set(gse111175_scores.columns) & set(asd_scores.columns))
    print(f'Shared pathways for projection: {len(shared_pathways)}/{gse111175_scores.shape[1]}')

    if len(shared_pathways) < 5:
        print('ERROR: Too few shared pathways for meaningful projection.')
        has_gse111175 = False
    else:
        # Fit GMM on GSE111175 data (shared pathways only)
        gse111175_k = gse111175_meta['subtype'].nunique()
        print(f'Fitting GMM (k={gse111175_k}) on GSE111175 shared-pathway scores...')

        gmm_reference = GaussianMixture(
            n_components=gse111175_k,
            covariance_type='full',
            random_state=SEED,
            n_init=10,
        )
        gmm_reference.fit(gse111175_scores[shared_pathways].values)

        # Verify reference model reproduces GSE111175 labels
        ref_labels = gmm_reference.predict(gse111175_scores[shared_pathways].values)
        ref_ari = adjusted_rand_score(gse111175_meta['subtype'].values, ref_labels)
        print(f'Reference model self-ARI: {ref_ari:.4f} (should be ~1.0)')

        # Project GSE18123 ASD samples into GSE111175 subtype space
        print(f'\nProjecting {len(asd_scores)} GSE18123 ASD samples...')
        projected_labels = gmm_reference.predict(asd_scores[shared_pathways].values)
        projected_probs = gmm_reference.predict_proba(asd_scores[shared_pathways].values)

        # Compare projected labels vs de novo labels
        projection_ari = adjusted_rand_score(labels, projected_labels)
        print(f'\n=== Cross-Cohort Projection Results ===')
        print(f'  De novo subtypes (GSE18123): k={optimal_k}')
        print(f'  Reference subtypes (GSE111175): k={gse111175_k}')
        print(f'  Projection ARI: {projection_ari:.4f}')
        threshold = 0.3
        status = 'PASS' if projection_ari > threshold else 'FAIL'
        print(f'  Threshold (ARI > {threshold}): {status}')

        # Projected subtype sizes
        print(f'\nProjected subtype distribution (GSE18123 → GSE111175 space):')
        for i in range(gse111175_k):
            count = (projected_labels == i).sum()
            print(f'  Projected subtype {i}: {count} ({count/len(projected_labels)*100:.1f}%)')

        # Mean prediction confidence
        max_probs = projected_probs.max(axis=1)
        print(f'\nProjection confidence: mean={max_probs.mean():.3f}, '
              f'median={np.median(max_probs):.3f}, min={max_probs.min():.3f}')

        asd_meta['projected_subtype'] = projected_labels
        asd_meta['projection_confidence'] = max_probs
else:
    print('Skipping cross-cohort projection (no GSE111175 data).')

In [ ]:
if has_gse111175:
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    # Reuse PCA embedding from Section 7
    # Left: De novo subtypes
    scatter_colors = plt.cm.Set2(np.linspace(0, 1, max(optimal_k, gse111175_k)))
    for i in range(optimal_k):
        mask = labels == i
        axes[0].scatter(embedding[mask, 0], embedding[mask, 1], c=[scatter_colors[i]],
                        label=f'De novo {i} (n={mask.sum()})', s=50, alpha=0.7,
                        edgecolors='k', linewidth=0.3)
    axes[0].set_title(f'De Novo Subtypes (k={optimal_k})')
    axes[0].set_xlabel('PC1')
    axes[0].set_ylabel('PC2')
    axes[0].legend(fontsize=8)

    # Middle: Projected subtypes
    for i in range(gse111175_k):
        mask = projected_labels == i
        axes[1].scatter(embedding[mask, 0], embedding[mask, 1], c=[scatter_colors[i]],
                        label=f'Projected {i} (n={mask.sum()})', s=50, alpha=0.7,
                        edgecolors='k', linewidth=0.3)
    axes[1].set_title(f'Projected from GSE111175 (k={gse111175_k})')
    axes[1].set_xlabel('PC1')
    axes[1].legend(fontsize=8)

    # Right: Projection confidence
    sc = axes[2].scatter(embedding[:, 0], embedding[:, 1], c=max_probs,
                         cmap='RdYlGn', s=50, alpha=0.8, edgecolors='k', linewidth=0.3,
                         vmin=0.3, vmax=1.0)
    plt.colorbar(sc, ax=axes[2], label='Projection Confidence')
    axes[2].set_title('Projection Confidence')
    axes[2].set_xlabel('PC1')

    plt.suptitle(f'GSE18123 → GSE111175 Cross-Cohort Projection (ARI={projection_ari:.3f})',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'cross_cohort_projection.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # Confusion-style cross-tabulation
    print('\n--- De Novo vs Projected Subtype Cross-Tabulation ---')
    print(pd.crosstab(
        pd.Series(labels, name='De_Novo'),
        pd.Series(projected_labels, name='Projected'),
        margins=True
    ))
else:
    print('Skipping projection visualization.')

In [ ]:
# Save cross-cohort projection results
if has_gse111175:
    import json as _json

    projection_results = {
        'reference_dataset': 'GSE111175',
        'target_dataset': 'GSE18123',
        'reference_k': int(gse111175_k),
        'target_denovo_k': int(optimal_k),
        'n_shared_pathways': len(shared_pathways),
        'shared_pathways': shared_pathways,
        'reference_self_ari': float(ref_ari),
        'projection_ari': float(projection_ari),
        'projection_threshold': 0.3,
        'projection_passed': bool(projection_ari > 0.3),
        'mean_confidence': float(max_probs.mean()),
        'projected_subtype_sizes': {
            str(i): int((projected_labels == i).sum())
            for i in range(gse111175_k)
        },
        'seed': SEED,
    }

    with open(os.path.join(OUTPUT_DIR, 'cross_cohort_projection_ari.json'), 'w') as f:
        _json.dump(projection_results, f, indent=2)

    print('Saved cross-cohort projection results.')
    print(f'  ARI: {projection_ari:.4f} ({"PASS" if projection_ari > 0.3 else "FAIL"})')
else:
    print('No projection results to save.')

## 10. Cross-Tissue Comparison with GSE28521 Brain

**Addressing Vulnerability V7** — postmortem brain only.

Compare the pathway enrichment profiles of blood-based subtypes (this notebook)
against the brain-based subtypes from GSE28521 (Notebook 10). If the same pathways
are enriched/depleted across tissues, it supports cross-tissue convergence.

In [ ]:
# Load GSE28521 frontal cortex results
from scipy import stats

has_brain_data = False

brain_search_paths = [
    ('research-results/GSE28521/frontal-cortex', 'local research-results'),
    ('../../research-results/GSE28521/frontal-cortex', 'parent research-results'),
    ('./outputs/gse28521/frontal_cortex', 'local outputs'),
]

brain_results_url = 'https://raw.githubusercontent.com/topmist-admin/pathway-subtyping-framework/main/examples/notebooks/outputs/gse28521/frontal_cortex'

for base_path, source_label in brain_search_paths:
    scores_path = os.path.join(base_path, 'fc_pathway_scores.csv')
    meta_path = os.path.join(base_path, 'fc_sample_metadata_with_subtypes.csv')
    try:
        brain_scores = pd.read_csv(scores_path, index_col=0)
        brain_meta = pd.read_csv(meta_path, index_col=0)
        print(f'Loaded GSE28521 frontal cortex results from {source_label}:')
        print(f'  Brain samples: {len(brain_scores)} ({(brain_meta["diagnosis"] == "ASD").sum()} ASD)')
        print(f'  Brain pathways: {brain_scores.shape[1]}')
        print(f'  Brain subtypes: {brain_meta["subtype"].nunique()}')
        has_brain_data = True
        break
    except Exception:
        continue

# Try GitHub URL as last resort
if not has_brain_data:
    try:
        brain_scores = pd.read_csv(f'{brain_results_url}/fc_pathway_scores.csv', index_col=0)
        brain_meta = pd.read_csv(f'{brain_results_url}/fc_sample_metadata_with_subtypes.csv', index_col=0)
        print(f'Loaded GSE28521 frontal cortex results from GitHub')
        has_brain_data = True
    except Exception:
        pass

if not has_brain_data:
    print('Could not load brain data. Skipping cross-tissue comparison.')
    print('Run Notebook 10 first to generate outputs.')

In [ ]:
if has_brain_data:
    # Shared pathways between blood and brain
    blood_pathways = set(asd_scores.columns)
    brain_pathways_set = set(brain_scores.columns)
    shared = sorted(blood_pathways & brain_pathways_set)
    print(f'Shared pathways: {len(shared)}/{len(blood_pathways)} blood, {len(brain_pathways_set)} brain')

    # Mean pathway scores by tissue (ASD only)
    blood_asd_mean = asd_scores[shared].mean()
    brain_asd_mask = brain_meta['diagnosis'] == 'ASD'
    brain_asd_mean = brain_scores.loc[brain_asd_mask, shared].mean()

    # Spearman correlation of mean pathway profiles
    rho, p_val = stats.spearmanr(blood_asd_mean.values, brain_asd_mean.values)
    print(f'\nCross-tissue correlation of mean ASD pathway profiles:')
    print(f'  Spearman ρ = {rho:.4f}, p = {p_val:.4f}')
    sig = 'SIGNIFICANT' if p_val < 0.05 else 'not significant'
    print(f'  → {sig}')

    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Left: scatter of mean scores
    ax = axes[0]
    ax.scatter(blood_asd_mean.values, brain_asd_mean.values, s=80, alpha=0.8,
               edgecolors='k', linewidth=0.5, c='#3498db')
    for i, pw in enumerate(shared):
        ax.annotate(pw.replace('_', '\n'), (blood_asd_mean.iloc[i], brain_asd_mean.iloc[i]),
                    fontsize=6, ha='center', va='bottom')
    ax.set_xlabel('Blood Mean Pathway Score (GSE18123)')
    ax.set_ylabel('Brain Mean Pathway Score (GSE28521 FC)')
    ax.set_title(f'Cross-Tissue Concordance\nSpearman ρ={rho:.3f}, p={p_val:.4f}')
    z = np.polyfit(blood_asd_mean.values, brain_asd_mean.values, 1)
    x_line = np.linspace(blood_asd_mean.min(), blood_asd_mean.max(), 100)
    ax.plot(x_line, np.polyval(z, x_line), 'r--', alpha=0.5)

    # Right: side-by-side bar chart (Z-normalized)
    ax2 = axes[1]
    x = np.arange(len(shared))
    width = 0.35
    blood_z = (blood_asd_mean - blood_asd_mean.mean()) / blood_asd_mean.std()
    brain_z = (brain_asd_mean - brain_asd_mean.mean()) / brain_asd_mean.std()
    ax2.barh(x - width/2, blood_z.values, width, label='Blood (GSE18123)', color='#e74c3c', alpha=0.7)
    ax2.barh(x + width/2, brain_z.values, width, label='Brain (GSE28521 FC)', color='#3498db', alpha=0.7)
    ax2.set_yticks(x)
    ax2.set_yticklabels([p.replace('_', ' ') for p in shared], fontsize=8)
    ax2.set_xlabel('Z-normalized Mean Score')
    ax2.set_title('Pathway Profiles: Blood vs Brain')
    ax2.legend(fontsize=9)
    ax2.axvline(x=0, color='black', linewidth=0.5)

    plt.suptitle('Cross-Tissue Validation: Blood (GSE18123) vs Brain (GSE28521)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'blood_vs_brain_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Skipping cross-tissue comparison (no brain data loaded).')

## 11. Benchmark Comparison

Compare the framework's pathway-based GMM approach against alternative methods.

In [ ]:
# Run benchmark comparison on ASD samples
print('Running benchmark comparison...')
bench_result = run_benchmark_comparison(
    gene_burdens=asd_expression,
    pathway_scores=asd_scores,
    pathways=pathways,
    n_clusters=optimal_k,
    seed=SEED,
)

print('\n' + bench_result.format_report())

In [ ]:
# Visualize benchmark results
methods = list(bench_result.method_results.keys())
silhouettes = [bench_result.method_results[m].silhouette for m in methods]
runtimes = [bench_result.method_results[m].runtime_seconds for m in methods]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

colors = ['#2ecc71' if m == bench_result.best_method else '#3498db' for m in methods]
bars = ax1.barh(methods, silhouettes, color=colors)
ax1.set_xlabel('Silhouette Score (higher is better)')
ax1.set_title('Clustering Quality: Method Comparison')
for bar, val in zip(bars, silhouettes):
    ax1.text(max(bar.get_width() + 0.005, 0.01), bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=10)

ax2.barh(methods, runtimes, color='#9b59b6')
ax2.set_xlabel('Runtime (seconds)')
ax2.set_title('Computational Cost')
for bar, val in zip(ax2.patches, runtimes):
    ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.2f}s', va='center', fontsize=10)

plt.suptitle(f'GSE18123 Blood: Benchmark Comparison (k={optimal_k})',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'benchmark_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Compare all clustering algorithms
algo_comparison = compare_algorithms(
    data=asd_scores.values,
    n_clusters=optimal_k,
    seed=SEED,
)

print(f'Most stable algorithm: {algo_comparison.most_stable_algorithm}')
print(f'\nPairwise ARI (inter-algorithm agreement):')
for pair, ari in algo_comparison.pairwise_ari.items():
    print(f'  {pair}: {ari:.4f}')

print(f'\nPer-algorithm metrics:')
for algo, res in algo_comparison.results.items():
    print(f'  {algo}: silhouette={res.silhouette:.4f}, CH={res.calinski_harabasz:.1f}, '
          f'DB={res.davies_bouldin:.4f}')

## 12. Summary & Export

Save all results for downstream use and generate the final summary.

In [ ]:
# Save key outputs
asd_scores.to_csv(os.path.join(OUTPUT_DIR, 'pathway_scores_asd.csv'))
pathway_scores.to_csv(os.path.join(OUTPUT_DIR, 'pathway_scores_all.csv'))
asd_meta.to_csv(os.path.join(OUTPUT_DIR, 'sample_metadata_with_subtypes.csv'))
gene_expression.to_csv(os.path.join(OUTPUT_DIR, 'gene_expression_processed.csv'))

# Save comprehensive results JSON
import json

results_summary = {
    'dataset': 'GSE18123',
    'citation': 'Kong et al. 2012, PLoS ONE',
    'tissue': 'peripheral blood',
    'platforms': list(platform_counts.keys()),
    'n_total_samples': int(len(pathway_scores)),
    'n_asd': int(asd_mask.sum()),
    'n_control': int((~asd_mask).sum()),
    'n_genes': int(gene_expression.shape[1]),
    'n_pathways_scored': int(scoring_result.n_pathways_scored),
    'scoring_method': 'ssGSEA',
    'optimal_k': int(optimal_k),
    'clustering_algorithm': 'GMM',
    'silhouette': float(clustering.silhouette),
    'calinski_harabasz': float(clustering.calinski_harabasz),
    'davies_bouldin': float(clustering.davies_bouldin),
    'validation_all_passed': bool(val_result.all_passed),
    'validation_gates': [
        {'name': str(g.name), 'passed': bool(g.passed), 'metric': str(g.metric_name),
         'value': float(g.metric_value), 'threshold': float(g.threshold)}
        for g in val_result.results
    ],
    'benchmark_best_method': str(bench_result.best_method),
    'benchmark_ranking': [str(r) for r in bench_result.ranking],
    'subtype_sizes': {str(i): int((labels == i).sum()) for i in range(optimal_k)},
    'vulnerabilities_addressed': ['V5_sample_size', 'V7_blood_tissue'],
    'framework_version': '0.3.0',
    'seed': SEED,
}

if has_gse111175:
    results_summary['cross_cohort_projection'] = {
        'reference_dataset': 'GSE111175',
        'projection_ari': float(projection_ari),
        'passed': bool(projection_ari > 0.3),
        'mean_confidence': float(max_probs.mean()),
    }

if has_brain_data:
    results_summary['cross_tissue'] = {
        'brain_dataset': 'GSE28521',
        'spearman_rho': float(rho),
        'spearman_p': float(p_val),
    }

with open(os.path.join(OUTPUT_DIR, 'results_summary.json'), 'w') as f:
    json.dump(results_summary, f, indent=2)

print('Results saved.')

In [ ]:
print('\n' + '=' * 60)
print('ANALYSIS COMPLETE')
print('=' * 60)
print(f'\nDataset: GSE18123 (Kong et al. 2012, PLoS ONE)')
print(f'Tissue: Peripheral blood (males, two Affymetrix platforms)')
print(f'Samples: {len(pathway_scores)} total ({asd_mask.sum()} ASD, {(~asd_mask).sum()} Control)')
print(f'Genes: {gene_expression.shape[1]} → {scoring_result.n_pathways_scored} pathways (ssGSEA)')
print(f'Optimal subtypes: {optimal_k} (BIC-selected, GMM)')
print(f'Silhouette: {clustering.silhouette:.4f}')
print(f'Validation: {"ALL PASSED" if val_result.all_passed else "SOME FAILED"}')
for gate in val_result.results:
    status = 'PASS' if gate.passed else 'FAIL'
    print(f'  [{status}] {gate.name}: {gate.metric_value:.4f}')
print(f'Best method: {bench_result.best_method}')

print(f'\n--- Cross-Cohort Projection ---')
if has_gse111175:
    print(f'  GSE111175 → GSE18123 ARI: {projection_ari:.4f} '
          f'({"PASS" if projection_ari > 0.3 else "FAIL"}, threshold > 0.3)')
    print(f'  Mean projection confidence: {max_probs.mean():.3f}')
else:
    print(f'  Not available (GSE111175 results not loaded)')

print(f'\n--- Cross-Tissue Comparison ---')
if has_brain_data:
    print(f'  Blood vs Brain Spearman ρ = {rho:.4f}, p = {p_val:.4f}')
else:
    print(f'  Not available (GSE28521 results not loaded)')

print(f'\n--- Vulnerabilities Addressed ---')
print(f'  V5 (sample size): n={asd_mask.sum()} ASD (9x larger than GSE28521 FC)')
print(f'  V7 (tissue):      Blood-based subtyping (second independent cohort)')

print(f'\nOutputs saved to: {OUTPUT_DIR}/')
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f'  {f}')

---

## References

1. Kong SW, et al. (2012). Characteristics and Predictive Value of Blood Transcriptome Signature in Males with Autism Spectrum Disorders. *PLoS ONE*, 7(12):e49475. [PMID: 23227143](https://pubmed.ncbi.nlm.nih.gov/23227143/)
2. Gazestani VH, et al. (2019). A perturbed gene network containing PI3K-AKT, RAS-ERK and WNT-β-catenin pathways in leukocytes is linked to ASD genetics and symptom severity. *Nature Neuroscience*, 22:1624-1634. [PMID: 31551594](https://pubmed.ncbi.nlm.nih.gov/31551594/)
3. Voineagu I, et al. (2011). Transcriptomic analysis of autistic brain reveals convergent molecular pathology. *Nature*, 474(7351):380-384. [PMID: 21614001](https://pubmed.ncbi.nlm.nih.gov/21614001/)
4. Satterstrom FK, et al. (2020). Large-Scale Exome Sequencing Study Implicates Both Developmental and Functional Changes in the Neurobiology of Autism. *Cell*, 180(3):568-584. [PMID: 31981491](https://pubmed.ncbi.nlm.nih.gov/31981491/)
5. Chauhan R (2026). Pathway Subtyping Framework v0.3.0. *Zenodo*. [DOI: 10.5281/zenodo.18442426](https://doi.org/10.5281/zenodo.18442426)

## Data Availability

- **GSE18123:** https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE18123
- **GSE111175:** https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE111175
- **GSE28521:** https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE28521
- **Framework:** https://github.com/topmist-admin/pathway-subtyping-framework
- **PyPI:** `pip install pathway-subtyping`

## License

This notebook is released under CC-BY 4.0.